# Actividad | Estimación de máxima verosimilitud y regresión lineal generalizada

* Luisa Merlo García | A01067715
* Edgar Osvaldo Navarro | A01644488
* Ariana Guadalupe Rosales Villalobos | A01644773

## Ejercicio 1 (30 puntos)

Considere la distribución binomial, la cual para un número entero $m$ mayor o igual que cero y una probabilidad de éxito $p$, la probabilidad de $P(X = x)$ está dada por:

$$P(X = x) = \binom{m}{x} p^x (1 - p)^{m-x}$$

1. Indique qué uso tiene esta distribución de probabilidad, y qué significado tiene $P(X = x)$.
2. Para una muestra aleatoria $\{x_1, x_2, x_3, \dots, x_n\}$ de tamaño $n$ que proviene de esta distribución, escriba las funciones de verosimilitud y log-verosimilitud correspondientes en términos de los parámetros $p$, $n$ y $m$.
3. Obtenga el estimador de máxima verosimilitud para el parámetro $p$ derivando la función de log-verosimilitud.
4. A través de simulaciones en Python, muestre si el estimador es sesgado o insesgado. Para estas simulaciones pruebe para distintos valores de $p$, $n$ y $m$ (al menos 6 combinaciones).

## Ejercicio 2 (30 puntos)

Durante la Segunda Guerra Mundial, los aliados intentaron estimar la cantidad de tanques producidos por Alemania a partir de los números de serie de los tanques capturados. Supongamos que un ejército ha fabricado $N$ tanques, numerados consecutivamente desde 1 hasta $N$, donde $N$ es desconocido. Se selecciona una muestra aleatoria de $n$ tanques, sin reemplazo, y se registran sus números de serie:

$$x_1, x_2, x_3, \dots, x_n$$

Se considera que todas las muestras posibles de tamaño ($n$) tienen la misma probabilidad de ser seleccionadas.

Responde los siguientes incisos:

1. Determina la probabilidad de observar una muestra específica de $N$ tanques en función del número total . Considera que el orden de selección no importa.
2. Construye la función de verosimilitud $L(N)$, indicando los valores que puede tomar el parámetro $N$. Define $m = \max(x_1, x_2, \dots, x_n)$.
3. Encuentra el estimador de máxima verosimilitud de $N$. Justifica matemáticamente por qué el valor propuesto maximiza la función de verosimilitud.
4. Supongamos que se capturan ocho tanques con los siguientes números de serie: 18, 47, 63, 91, 105, 132, 156, 184.
   - Calcula la estimación de máxima verosimilitud de $N$.
   - Grafica la función de verosimilitud para valores de entre 184 y 400, normalizándola respecto a su valor máximo.
5. Explica si consideras que el estimador obtenido tiende a sobreestimar o subestimar la cantidad real de tanques. Justifica tu respuesta a partir de simulaciones.
6. Para este problema, se sabe que el valor esperado del máximo muestral es $\hat{N} = \frac{n+1}{n} m - 1$. Utiliza esta expresión para construir un estimador insesgado de $N$. Calcula su valor para la muestra del inciso 4 y compara los resultados de ambos estimadores.
7. ¿Por qué el estimador de máxima verosimilitud y el estimador insesgado producen resultados diferentes? ¿Es necesariamente preferible un estimador insesgado a uno sesgado? Justifica tu respuesta.

## Ejercicio 3

El conjunto de datos de [cangrejos hembra](#) contiene las siguientes variables:

* **sat** = número de cangrejos macho satélites
* **y** = 0 indica que no tiene cangrejos satélite la hembra, 1 indica que si hay cangrejos satélites
* **weight** = peso del cangrejo
* **width** = ancho del cangrejo hembra
* **color** = color del cangrejo hembra
* **spine** = número de púas del cangrejo hembra

1. Si queremos un modelo para la variable dependiente **sat**, con predictores **weight**, **width**, **color** y **spine**, ¿qué modelo de regresión lineal generalizado sería el más apropiado? ¿Por qué?
2. Ajuste el modelo que seleccionaste para la variable **sat**.
3. ¿Qué predictores quedarían dentro del modelo de acuerdo a los p-valores indicados al momento de hacer la regresión?
4. Si queremos un modelo para la variable dependiente **y**, con predictores **weight**, **width**, **color** y **spine**, ¿qué modelo de regresión lineal generalizado sería el más apropiado? ¿Por qué?
5. Ajuste el modelo que seleccionaste para la variable **y**.
6. ¿Qué predictores quedarían dentro del modelo de acuerdo a los p-valores indicados al momento de hacer la regresión?

In [1]:
import pandas as pd, numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
d = pd.read_csv('crab.csv')
print(d.shape)
print(d[['sat','y']].describe().loc[['mean','std']])
print('Media de sat:', d.sat.mean().round(3), '| Varianza de sat:', d.sat.var().round(3))
print(d.groupby('y').sat.agg(['count','min','max']))
print('y=1 exactamente cuando sat>0:', ((d.sat>0).astype(int)==d.y).all())

(173, 6)
           sat         y
mean  2.919075  0.641618
std   3.148336  0.480917
Media de sat: 2.919 | Varianza de sat: 9.912
   count  min  max
y                 
0     62    0    0
1    111    1   15
y=1 exactamente cuando sat>0: True


## Inciso 1
El modelo más apropiado es una regresión de Poisson con liga log. La variable `sat` es un conteo (enteros desde 0) y la liga log garantiza que la media estimada sea positiva. Un modelo lineal normal podría predecir valores negativos y supone varianza constante, lo cual no ocurre en un conteo.

La varianza de `sat` (9.9) es mayor que su media (2.9), lo que indica posible sobredispersión. Esto se verifica después del inciso 3.

## Inciso 2
Ajuste del modelo de Poisson con los cuatro predictores.

In [2]:
m1 = smf.glm('sat ~ weight + width + color + spine', data=d, family=sm.families.Poisson()).fit()
print(m1.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                    sat   No. Observations:                  173
Model:                            GLM   Df Residuals:                      168
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -453.58
Date:                Mon, 02 Nov 2026   Deviance:                       551.85
Time:                        12:30:46   Pearson chi2:                     545.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.3737
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.5285      0.949     -0.557      0.5

## Inciso 3
P-valores del resumen del modelo, con alfa = 0.05.

In [3]:
p = m1.pvalues.drop('Intercept')
res = pd.DataFrame({'coef':m1.params.drop('Intercept'),'p-valor':p,'significativo (0.05)':p<0.05})
print(res.round(4))
print('Predictores que quedan en el modelo:', list(p[p<0.05].index))

          coef  p-valor  significativo (0.05)
weight  0.4725   0.0042                  True
width   0.0275   0.5659                 False
color  -0.1849   0.0054                  True
spine   0.0400   0.4816                 False
Predictores que quedan en el modelo: ['weight', 'color']


Según los p-valores del ajuste de Poisson, los predictores que permanecen en el modelo son `weight` (p = 0.004) y `color` (p = 0.005). Los predictores `width` (p = 0.57) y `spine` (p = 0.48) no son significativos.

El cociente de Pearson entre los grados de libertad es 3.24, mayor que 1, por lo que los errores estándar de Poisson están subestimados. Con un ajuste quasi-Poisson o binomial negativo los p-valores aumentan y ningún predictor resulta significativo.

In [4]:
print('Pearson chi2 / gl =', round(m1.pearson_chi2/m1.df_resid,3))
mq = smf.glm('sat ~ weight + width + color + spine', data=d, family=sm.families.Poisson()).fit(scale='X2')
mnb = smf.negativebinomial('sat ~ weight + width + color + spine', data=d).fit(disp=0)
print(pd.DataFrame({'Poisson':m1.pvalues,'quasi-Poisson':mq.pvalues,'Bin. negativa':mnb.pvalues.drop('alpha')}).round(4))

Pearson chi2 / gl = 3.242
           Poisson  quasi-Poisson  Bin. negativa
Intercept   0.5775         0.7570         0.7208
weight      0.0042         0.1115         0.1111
width       0.5659         0.7498         0.8454
color       0.0054         0.1226         0.1557
spine       0.4816         0.6959         0.8906


## Inciso 4
El modelo más apropiado es una regresión logística (binomial con liga logit). La variable `y` toma solo los valores 0 y 1, y la liga logit mantiene las probabilidades estimadas entre 0 y 1. Los coeficientes se interpretan como log-odds.

## Inciso 5
Ajuste del modelo logístico para `y`.

In [5]:
m2 = smf.glm('y ~ weight + width + color + spine', data=d, family=sm.families.Binomial()).fit()
print(m2.summary())
print('Odds ratios:'); print(np.exp(m2.params).round(4))

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                  173
Model:                            GLM   Df Residuals:                      168
Model Family:                Binomial   Df Model:                            4
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -93.330
Date:                Mon, 02 Nov 2026   Deviance:                       186.66
Time:                        12:30:46   Pearson chi2:                     170.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2023
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -7.5994      3.754     -2.024      0.0

## Inciso 6

In [6]:
p2 = m2.pvalues.drop('Intercept')
res2 = pd.DataFrame({'coef':m2.params.drop('Intercept'),'p-valor':p2,'significativo (0.05)':p2<0.05})
print(res2.round(4))
print('Predictores que quedan en el modelo:', list(p2[p2<0.05].index))

          coef  p-valor  significativo (0.05)
weight  0.7949   0.2505                 False
width   0.2733   0.1490                 False
color  -0.5915   0.0144                  True
spine   0.2717   0.2597                 False
Predictores que quedan en el modelo: ['color']


El único predictor que permanece en el modelo es `color` (p = 0.014, coeficiente de -0.59, es decir, a mayor valor de color disminuye la probabilidad de tener satélites). Los predictores `weight` (p = 0.25), `width` (p = 0.15) y `spine` (p = 0.26) no son significativos. Cabe mencionar que `weight` y `width` tienen una correlación de 0.887, lo que aumenta sus errores estándar.

In [7]:
print(d[['weight','width']].corr().round(3))

        weight  width
weight   1.000  0.887
width    0.887  1.000
